# 07 — EvidenceGraph & Multi-Engine Architecture

Demonstrates the shared explainability IR:
- `EvidenceGraph` data model: `EvidenceNode`, `EvidenceEdge`
- Two layout modes: **tree** (Souffle/ProbLog) and **timeline** (PyReason)
- HTML renderer for both layouts
- Round-trip serialization (`to_dict` / `from_dict`)
- Architecture summary: unified IR, not unified implementation

This notebook uses **no real engine calls** — all examples are manually constructed
to illustrate the shared data model.

**Prerequisites:** [01](01_sdk_basics.ipynb)–[06](06_problog_probabilistic.ipynb) for context.

## 0. Imports

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

In [2]:
from factpy_kernel.audit.evidence_graph import (
    EvidenceGraph, EvidenceNode, EvidenceEdge,
    LAYOUT_TREE, LAYOUT_TIMELINE,
    NODE_CONCLUSION, NODE_PREMISE, NODE_SEED,
    EDGE_SUPPORTS, EDGE_DERIVES, EDGE_UPDATES,
    evidence_graph_to_dict, evidence_graph_from_dict,
    render_evidence_graph_html,
)
from IPython.display import HTML, display

## 1. EvidenceGraph Data Model

```
EvidenceGraph
  ├── graph_id, engine, root_node_id, support_kind
  ├── layout_hint: LAYOUT_TREE | LAYOUT_TIMELINE
  ├── metadata: dict
  ├── nodes: tuple[EvidenceNode, ...]
  │     ├── node_id, node_kind (conclusion/premise/seed)
  │     ├── component, label, value_summary
  │     ├── timestamp (timeline only)
  │     └── engine_meta: dict
  └── edges: tuple[EvidenceEdge, ...]
        ├── edge_id, source_id, target_id
        ├── edge_kind (supports/derives/updates)
        └── rule_label
```

## 2. Tree Layout (Souffle / ProbLog Style)

Tree layout renders as a recursive parent-child structure.
Used when the engine produces a proof tree (deductive derivation).

In [3]:
tree_graph = EvidenceGraph(
    graph_id="demo:tree", engine="problog",
    root_node_id="n:root", support_kind="problog_provenance_v1",
    layout_hint=LAYOUT_TREE,
    metadata={"probability": 0.85},
    nodes=(
        EvidenceNode("n:root", NODE_CONCLUSION, "Alice", "expertise", "0.85"),
        EvidenceNode("n:p1", NODE_PREMISE, "Alice", "name", "Alice Chen"),
        EvidenceNode("n:p2", NODE_PREMISE, "Alice", "publications", "42"),
    ),
    edges=(
        EvidenceEdge("e:1", "n:p1", "n:root", EDGE_DERIVES, rule_label="expertise_rule"),
        EvidenceEdge("e:2", "n:p2", "n:root", EDGE_SUPPORTS),
    ),
)

print(f"Tree graph: {len(tree_graph.nodes)} nodes, {len(tree_graph.edges)} edges")
print(f"  layout: {tree_graph.layout_hint}")
print(f"  root: {tree_graph.root_node_id}")

display(HTML(render_evidence_graph_html(tree_graph)))

Tree graph: 3 nodes, 2 edges
  layout: tree
  root: n:root


## 3. Timeline Layout (PyReason Style)

Timeline layout renders as a CSS grid with timestep columns and component rows.
Used when the engine produces an event log (temporal propagation).

In [4]:
timeline_graph = EvidenceGraph(
    graph_id="demo:timeline", engine="pyreason",
    root_node_id="n:t1", support_kind="pyreason_provenance_v1",
    layout_hint=LAYOUT_TIMELINE,
    metadata={"timesteps": 2},
    nodes=(
        EvidenceNode("n:t0", NODE_SEED, "Alice", "impact", "[1.0, 1.0]", timestamp=0),
        EvidenceNode("n:t1", NODE_CONCLUSION, "Bob", "impact", "[0.8, 0.9]", timestamp=1,
                     engine_meta={"old_bound": [0, 1], "new_bound": [0.8, 0.9]}),
    ),
    edges=(
        EvidenceEdge("e:1", "n:t0", "n:t1", EDGE_DERIVES, rule_label="impact_propagation"),
    ),
)

print(f"Timeline graph: {len(timeline_graph.nodes)} nodes, {len(timeline_graph.edges)} edges")
print(f"  layout: {timeline_graph.layout_hint}")

display(HTML(render_evidence_graph_html(timeline_graph)))

Timeline graph: 2 nodes, 1 edges
  layout: timeline


## 4. Serialization Round-Trip

`EvidenceGraph` serializes to/from dict for storage in `evidence_graphs.jsonl`.

In [5]:
d = evidence_graph_to_dict(tree_graph)
print("Serialized keys:", list(d.keys()))
print(f"  nodes: {len(d['nodes'])}, edges: {len(d['edges'])}")

restored = evidence_graph_from_dict(d)
print(f"\nRound-trip OK: {restored.graph_id == tree_graph.graph_id}")
print(f"  nodes match: {len(restored.nodes) == len(tree_graph.nodes)}")
print(f"  edges match: {len(restored.edges) == len(tree_graph.edges)}")

Serialized keys: ['graph_id', 'engine', 'root_node_id', 'nodes', 'edges', 'support_kind', 'layout_hint', 'metadata']
  nodes: 3, edges: 2

Round-trip OK: True
  nodes match: True
  edges match: True


## 5. Three-Layer Explain Architecture

The framework provides three explain layers, each serving a different consumer:

**Layer 1: Engine-native provenance** (highest fidelity)
```
Souffle:  SouffleProofTreeV0   — proof tree (lossless engine output)
ProbLog:  ProbLogTraceV0       — proof trace (call frame tree)
PyReason: PyReasonTraceV0      — event log (bound updates per timestep)
```

**Layer 2: Runtime explain contracts** (engine-specific runtime surface)

| Engine | Contract | Endpoint | 状态 |
|--------|----------|----------|------|
| native/souffle | `CandidateEvidenceTree` | `explain-tree` | ✅ |
| pyreason | `CandidateProvenanceTimeline` | `explain-timeline` | ✅ |
| problog | `CandidateEvidenceTree` | `explain-tree` | ✅ (accepted candidates, proof_goal/proof_leaf node kinds) |

ProbLog 的 trace 本身是 tree 形（call frame tree），自然方向是接入 `CandidateEvidenceTree`（和 native/souffle 相同），而不是新建合同。
PyReason 则是 event log 形态，不是 tree，因此需要独立的 `CandidateProvenanceTimeline` 合同。

**Layer 3: EvidenceGraph** (unified audit/static visualization)
```
Souffle:  SouffleProofTreeV0  →  souffle_proof_tree_to_evidence_graph()  →  EvidenceGraph(tree)
ProbLog:  ProbLogTraceV0      →  problog_trace_to_evidence_graph()       →  EvidenceGraph(tree)
PyReason: PyReasonTraceV0     →  pyreason_trace_to_evidence_graph()      →  EvidenceGraph(timeline)
```

Key principle: **unified interface, not unified implementation.**
- Layer 1 preserves full engine fidelity
- Layer 2 provides engine-appropriate runtime explain (tree OR timeline, not forced into one shape)
- Layer 3 provides cross-engine audit/static rendering via a shared DTO

## 6. Framework Architecture Summary

```
SDK Layer:  Entity / Relationship / Rule(engine_ext=...) / Derivation
    │
Core Layer: Store.evaluate(mode=..., engine_options=...) → CandidateSet
    │       Ledger: Claim + AnnotationRow (dual-write) + MetaRow (legacy)
    │
Adapter Layer:
    ├── Souffle:  proof tree   → CandidateEvidenceTree (runtime)
    │                          → EvidenceGraph(tree) (audit)
    ├── ProbLog:  --trace      → CandidateEvidenceTree (runtime, proof_goal/proof_leaf)
    │                          → EvidenceGraph(tree) (audit)
    └── PyReason: event log    → CandidateProvenanceTimeline (runtime)
                               → EvidenceGraph(timeline) (audit)
    │
Audit Layer: evidence_graphs.jsonl + provenance_timelines.jsonl
    │         render_evidence_graph_html() → unified HTML viewer
    │
Principles:
    - probability / bound / active_from are fact semantic properties
    - engine_ext = definition-time, engine_options = call-time
    - Runtime explain is engine-appropriate (tree vs. timeline)
    - EvidenceGraph is the shared audit IR, not the runtime explain surface
    - Unified interface, not unified implementation
```